## Optical signal propagation during an earthquake
The other demos demonstrate signal transmission and propagation, and earthquake modelling.
This notebook brings those two together, modelling the propagation of a signal during an earthquake.

We start by defining the system parameters as usual.

In [ ]:
from configparser import ConfigParser

parameters = ConfigParser()

parameters['FIBRE'] = {
    'section_length':    '1',  # Correlation length in km
    'path_coordinates': '[ \
        [-73.10897397357299,-36.83168432227461], \
        [-73.12487396171335,-35.95811819864919], \
        [-72.8998741217013,-35.47096027456558], \
        [-72.41606446443656,-35.3183747616349], \
        [-72.44987444048532,-34.73466151270862], \
        [-72.22487459987728,-33.991743556435], \
        [-71.60009504247698,-33.54688527485856] \
    ]',                          # Estimate coordinates of the Prat cable, taken from https://www.submarinecablemap.com/api/v3/cable/cable-geo.json
    'PMD_parameter':     '0.1',  # Polarisation mode dispersion parameter in ps / (km ^ 0.5)
    'realisation_count': '1000', # Number of fibre realisations to simulate simultaneously
    'photoelasticity':   '0.78', # Photoelasticity, which relates material strain to optical strain
    'group_delay':       '4.9e6' # Group delay in ps / km
}

parameters['TRANSCEIVER'] = {
    'constellation':    'QPSK',  # The symbol constellation to use
    'power':            '2',     # Transmission power in dBm
    'baud_rate':        '1e11',  # Baud rate in symbols / s
    'pulse':            'RRCOS', # Pulseshape, can be SINC or RRCOS, or define your own using the Pulse class
    'pulse_parameter':  '0.5',   # Parameter to pass to the pulse constructor. For a RRCOS pulse, this is the rolloff factor
    'upsample_factor':  '4',     # Samples per symbol
    'filter':           'RRCOS', # Antialiasing (matched) filter
    'filter_parameter': '0.5'    # Same
}

parameters['SIGNAL'] = {
    'batch_size':   '1', # The number of signals to transmit simultaneously
    'symbol_count': '25' # The number of symbols to transmit per signal
}

parameters['EARTHQUAKE'] = {
    'event': 'GCMT:C201002270634A', # A historic earthquake event, structured <catalog>:<identifier> (e.g. from https://www.globalcmt.org/)
    'model': 'ak135f_5s'             # Earth model for Syngine to use from https://ds.iris.edu/ds/products/syngine/#earth
}

Now, we create the transceiver, fibre and earthquake.

In [ ]:
from tremor_waveplate_toolbox import Transmitter, Fibre, Receiver, Earthquake

transmitter = Transmitter(parameters)
fibre = Fibre(parameters)
receiver = Receiver(parameters)
earthquake = Earthquake(parameters)

Next, we obtain the strain imposed on the fibre by the earthquake over time

In [ ]:
earthquake_time, _, _, _, _, strain = earthquake(fibre, verbose = True)

We specify the moment at which transmission begins, relative to the start of the earthquake measurement.
As strain changes slowly with respect to the optical sample rate, we assume strain to be constant from this moment until the transmission end.
Signal time is corrected for the mean group delay, so we have to transform the time axis to obtain the correct corresponding strain.

In [ ]:
import numpy as np

transmission_time       = 360 # Start transmission one minute after the earthquake
earthquake_sample_index = np.searchsorted(earthquake_time, transmission_time) # Assume constant strain during signal transmission
sampled_strain          = strain[:, earthquake_sample_index]  # Strain during transmission
print(sampled_strain)

Now, let's apply this strain to the fibre and compare signal propagation to the strainless case.

In [ ]:
_, pilot_signal = transmitter.transmit_random(1, parameters.getint('SIGNAL', 'symbol_count'))
fibre.section_material_strain = np.zeros_like(fibre.section_material_strain)
received_signal_lol = fibre(pilot_signal)
received_signal = received_signal_lol.copy()

fibre.section_material_strain = sampled_strain
received_signal_strain = fibre(pilot_signal)

print(1 + fibre.section_material_strain)
print(1 + fibre.section_optical_strain)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize = (10, 10))

axes[0, 0].set_title("x polarisation")
axes[0, 0].set_ylabel("Real amplitude")
axes[0, 0].set_ylim([-0.04, 0.04])
axes[0, 0].grid()
axes[0, 0].plot(received_signal.time * 1e9, received_signal.samples_time[0, 0, :, 0].real)
axes[0, 0].plot(received_signal_strain.time * 1e9, received_signal_strain.samples_time[0, 0, :, 0].real)

axes[1, 0].set_xlabel("t [ns]")
axes[1, 0].set_ylabel("Imaginary amplitude")
axes[1, 0].set_ylim([-0.04, 0.04])
axes[1, 0].grid()
axes[1, 0].plot(received_signal.time * 1e9, received_signal.samples_time[0, 0, :, 0].imag)
axes[1, 0].plot(received_signal_strain.time * 1e9, received_signal_strain.samples_time[0, 0, :, 0].imag)

axes[0, 1].set_title("y polarisation")
axes[0, 1].set_ylim([-0.04, 0.04])
axes[0, 1].grid()
axes[0, 1].plot(received_signal.time * 1e9, received_signal.samples_time[0, 0, :, 1].real)
axes[0, 1].plot(received_signal_strain.time * 1e9, received_signal_strain.samples_time[0, 0, :, 1].real)

axes[1, 1].set_xlabel("t [ns]")
axes[1, 1].set_ylim([-0.04, 0.04])
axes[1, 1].grid()
axes[1, 1].plot(received_signal.time * 1e9, received_signal.samples_time[0, 0, :, 1].imag)
axes[1, 1].plot(received_signal_strain.time * 1e9, received_signal_strain.samples_time[0, 0, :, 1].imag)

fig.tight_layout()

plt.show()

In [ ]:
print(received_signal.samples_time - received_signal_strain.samples_time)